Random Forest-based Model for Real-time Detection of Freezing of Gait

This research project 

First Download the Data and convert from .txt to individual .csv files.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
# load .txt files
files = glob.glob("data/csv/daphnet/*.txt")
# create columns to a csv
'''
1. Time of sample in millisecond
2. Ankle (shank) acceleration - horizontal forward acceleration (mg)
3. Ankle (shank) acceleration - vertical (mg)
4. Ankle (shank) acceleration - horizontal lateral (mg)
5. Upper leg (thigh) acceleration - horizontal forward acceleration (mg)
6. Upper leg (thigh) acceleration - vertical (mg)
7. Upper leg (thigh) acceleration - horizontal lateral (mg)
8. Trunk acceleration - horizontal forward acceleration (mg)
9. Trunk acceleration - vertical (mg)
10. Trunk acceleration - horizontal lateral (mg)
11. Annotations (see Annotations section) 0 = Not part of experiment 1 = No Freeze 2 = Freeze

'''
columns = [
    'time', 
    'ankle_forward', 'ankle_vert', 'ankle_lat', 
    'thigh_forward', 'thigh_vert', 'thigh_lat',
    'trunk_forward', 'trunk_vert', 'trunk_lat',
    'label'
]

for i, file in enumerate(files):
    df = pd.read_csv(file, sep=' ', header=None, names=columns)
    
    # Save individual patient file
    patient_filename = f'data/csv/patient_{i:02d}.csv'
    os.makedirs('data/csv', exist_ok=True)
    df.to_csv(patient_filename, index=False)
    
    print(f"Patient {i}: {df.shape} - Labels: {df['label'].value_counts().to_dict()}")

print("Saved individual patient files to data/csv/")

Now, we need to pre-process by cleaning and filtering our data 

In [ ]:
from scipy.signal import butter, filtfilt
import numpy as np
# Overlapping of 0.5s (0-4, 0.5-4.5, 1-5)
# Defaults are cutoff after 20 hz, have a sampling rate of 64, window size of 6, and window overlap of 0.5
class DataPreprocessor:
    def __init__(self,cutoff=20, sampling_rate=64, window_size=6.0, window_overlap=0.5):
        self.cutoff = cutoff
        self.sampling_rate = sampling_rate
        self.window_size = window_size
        self.window_overlap = window_overlap
        
        
    def apply_butter(self, data):
        b, a = butter(N=4, Wn=self.cutoff, fs=self.sampling_rate, btype='low')
        filtered_data = filtfilt(b, a, data)
        return filtered_data
    
    
    def create_windows(self,data):
        # window iteration of 384 samples (6s * 64 hz) and overlap iter of 32 samples (0.5s * 64hz )
        windows = []
        window_size = int(self.window_size * self.sampling_rate)
        overlap_size = int(self.window_overlap * self.sampling_rate)
        for start in range(0, len(data) - window_size + 1, overlap_size):
            end = start + window_size
            windows.append(data[start:end])
        return np.array(windows)
    
    
    def normalize_windows(self, windows):
        means = windows.mean(axis=1, keepdims=True) 
        stds = windows.std(axis=1, keepdims=True)    
        stds = np.where(stds == 0, 1, stds)
        return (windows - means) / stds

    

After pre-processing our data, we can now look for key features to extract, depending on our domain knowledge

In [ ]:
from scipy import signal, stats
import numpy as np
import pandas as pd

class FeatureExtractor:
    def __init__(self, sampling_rate=64, expected_window_size = 6.0):
        self.sampling_rate = sampling_rate
        self.expected_window_size = int(expected_window_size * self.sampling_rate)
    def calculate_freq_features(self, window):
        # if inputted window less than 4 seconds
        if len(window) != self.expected_window_size:
            raise ValueError(f"Expected {self.expected_window_size} samples, got {len(window)}")
        f,psd = signal.welch(window, fs=self.sampling_rate)
        # Freeze power is TOTAL energy Between [3,8]
        freeze_power = psd[(f >=3) & (f <= 8)].sum()
        # Locomotion power is TOTAL energy Between [0.5,3]
        loco_power = psd[(f >= 0.5) & (f <= 3)].sum()
        freeze_index =  freeze_power/loco_power if loco_power > 0 else 0
        total_power = np.sum(psd)
        if total_power == 0 or np.isclose(total_power, 0):
            # no meaningful frequency content - assign default value
            spectral_centroid = 0.0  
        else:
            spectral_centroid = np.sum(f * psd) / total_power
        return freeze_index, spectral_centroid
    
    def calculate_energy(self, window):
        return np.sum(window**2)
    
    def calculate_var(self, window):
        return np.var(window)
    
    def calculate_skew(self,window):
        skew = stats.skew(window)
        if not np.isfinite(skew):
            skew = 0.0
        return skew
    
    def extract_features(self, windows):
        features_matrix = []
        for window in windows:
            freeze_index, spectral_centroid = self.calculate_freq_features(window)
            energy = self.calculate_energy(window)
            var = self.calculate_var(window)
            skew = self.calculate_skew(window)
            features_matrix.append([freeze_index, energy, var, skew, spectral_centroid])
        return np.array(features_matrix)

Now that we're able to successfuly preprocess and extract all the key features from the data, we can build a Random Forest class to start recognizing patterns in our data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import pandas as pd

class Model():
    def __init__(self, model_type='random_forest'):
        self.model_type = model_type
        if model_type =='random_forest':
            self.model = RandomForestClassifier(
                n_estimators=100,
                max_depth=8,
                min_samples_leaf=5,
                class_weight='balanced_subsample',
                random_state=42,
                n_jobs=-1
            )
            # decision tree
        else: 
            self.model = DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=42
            )
    # note do not need a train function as you are retraining everytime throughout LOSO (if you're using the same exact dataset, use train function to reduce redundant training calls)
    def loso(self, all_data):
        patient_results = []
        all_predictions = []
        all_probs = []
        all_labels = []
        with tqdm(total=len(all_data.keys()), desc=f"Running LOSO Cross-Validation") as pbar:
            for test_patient in all_data.keys():
                pbar.set_description(f"LOSO - Testing on patient {test_patient}")
                
                X_train_list = [all_data[p].iloc[:,:-1] for p in all_data if p != test_patient]
                y_train_list = [all_data[p].iloc[:,-1] for p in all_data if p != test_patient]
                
                X_train = pd.concat(X_train_list, ignore_index=True)
                y_train = pd.concat(y_train_list, ignore_index=True)
                
                # test set
                X_test = all_data[test_patient].iloc[:,:-1]
                y_test = all_data[test_patient].iloc[:,-1]
                
                # Predict
                scores, y_pred = self.train_and_predict(X_train, y_train, X_test)
                
                all_probs.extend(scores)
                all_predictions.extend(y_pred)
                all_labels.extend(y_test)
                
                # Calculating individual loso stats
                patient_metrics = self._calculate_metrics(y_test, y_pred)
                patient_results.append({
                    'patient_id': test_patient,
                    'sensitivity': patient_metrics['sensitivity'],
                    'specificity': patient_metrics['specificity'],
                    'n_windows': len(y_test),
                    'n_fog_windows': sum(y_test),
                    'y_true': list(y_test),
                    'y_pred': list(y_pred),
                    'y_proba': list(scores)
                })
                
                pbar.update(1)
            
        return all_labels, all_predictions, all_probs, patient_results
            
    # model functions
    def train_and_predict(self, X_train, y_train, X_test, threshold=0.45):
        self.model.fit(X_train, y_train)
        scores = self.model.predict_proba(X_test)[:, 1]
        predictions = (scores >= threshold).astype(int)
        return scores, predictions
    
    def get_feature_importance(self):
        return self.model.feature_importances_
    # model evaluations and data visualization
    def _calculate_metrics(self, y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        return {
            'sensitivity': sensitivity,
            'specificity': specificity,
            'confusion_matrix': (tn, fp, fn, tp)
        }


Finally, data visualization step

In [ ]:
from .preprocess import DataPreprocessor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from tqdm import tqdm
# data visualization imports
from sklearn.metrics import confusion_matrix,roc_curve, auc, classification_report, precision_recall_curve
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

class DataVisualizer:
    def __init__(self):
        plt.style.use('seaborn-v0_8-darkgrid')
        sns.set_palette("husl")
    def plot_raw_vs_processed_data(self, patient_file, preprocessor: DataPreprocessor, start_idx=0, duration= 30):
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        # Load data
        data = np.genfromtxt(patient_file, delimiter=',', skip_header=1)
        thigh_data = data[:,4]
        labels = data[:,-1]
        # Create a time array
        time = np.arange(len(thigh_data)) / preprocessor.sampling_rate
        # Select window to display
        display_samples = int(duration * preprocessor.sampling_rate)
        start_idx = start_idx
        end_idx = min(start_idx + display_samples, len(thigh_data))
        
        time_window = time[start_idx:end_idx]
        raw_window = thigh_data[start_idx:end_idx]
        labels_window = labels[start_idx:end_idx]
        # Apply Preprocessing
        filtered_data = preprocessor.apply_butter(thigh_data)
        filtered_window = filtered_data[start_idx:end_idx]
        # Plot Raw Signals
        ax1 = axes[0]
        ax1.plot(time_window, raw_window, 'b-', alpha=0.7, linewidth=0.8, label='Raw Signal')
        # Highlight FoG Events
        fog_mask = labels_window == 2
        if np.any(fog_mask):
            ax1.fill_between(time_window, ax1.get_ylim()[0], ax1.get_ylim()[1], 
                            where=fog_mask, alpha=0.3, color='red', label='FoG Event')
        ax1.set_ylabel('Acceleration (mg)')
        ax1.set_title('Raw Signal with FoG Events', fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        
         # Plot Preprocessed Signals
        ax2 = axes[1]
        ax2.plot(time_window, filtered_window, 'g-', alpha=0.7, linewidth=0.8, label='Filtered Signal')
        
        # Highlight FoG events
        if np.any(fog_mask):
            ax2.fill_between(time_window, ax2.get_ylim()[0], ax2.get_ylim()[1], 
                            where=fog_mask, alpha=0.3, color='red', label='FoG Event')
        
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel('Acceleration (mg)')
        ax2.set_title('Preprocessed Signal (Butterworth Filtered)', fontweight='bold')
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle(f'Signal Comparison (Window starting at {start_idx/preprocessor.sampling_rate:.1f}s)', 
                fontsize=14, fontweight='bold')
        plt.tight_layout()
        return fig
    
    
    def plot_confusion_matrix_roc_pr(self, y_true, y_pred, y_proba=None):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # making confusion matrix
        ax1 = axes[0]
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True,
                   xticklabels=['No FoG', 'FoG'], yticklabels=['No FoG', 'FoG'], ax=ax1)
        ax1.set_ylabel('True Label')
        ax1.set_xlabel('Predicted Label')
        ax1.set_title('Confusion Matrix', fontweight='bold')
        
        # calculate metrics
        
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        metrics_text = f'Sensitivity: {sensitivity:.1%}\nSpecificity: {specificity:.1%}\nAccuracy: {accuracy:.1%}'
        ax1.text(2.5, 0.5, metrics_text, fontsize=10, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # roc curve
        
        ax2 = axes[1]
        if y_proba is not None:
            fpr, tpr, _ = roc_curve(y_true, y_proba)
            roc_auc = auc(fpr, tpr)
            ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
            ax2.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Random')
            ax2.set_xlabel('False Positive Rate')
            ax2.set_ylabel('True Positive Rate')
            ax2.set_title('ROC Curve', fontweight='bold')
            ax2.legend(loc='lower right')
            ax2.grid(True, alpha=0.3)
        else:
            ax2.text(0.5, 0.5, 'No probability scores available\nfor ROC curve', 
                    ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title('ROC Curve (Not Available)', fontweight='bold')
        
        # pr curve
        ax3 = axes[2]
        if y_proba is not None:
            precision, recall, _ = precision_recall_curve(y_true, y_proba)
            ax3.plot(recall, precision, 'g-', linewidth=2)
            ax3.set_xlabel('Recall (Sensitivity)')
            ax3.set_ylabel('Precision')
            ax3.set_title('Precision-Recall Curve', fontweight='bold')
            ax3.grid(True, alpha=0.3)
            
            # add avg precision score
            from sklearn.metrics import average_precision_score
            avg_precision = average_precision_score(y_true, y_proba)
            ax3.text(0.1, 0.1, f'Avg Precision: {avg_precision:.3f}', 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        else:
            ax3.text(0.5, 0.5, 'No probability scores available\nfor PR curve', 
                    ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title('Precision-Recall Curve (Not Available)', fontweight='bold')
        
        plt.suptitle('Model Performance Metrics', fontsize=14, fontweight='bold')
        plt.tight_layout()
        return fig
    